# IBM HR Analytics — Employee Attrition
quick exploration of what drives people to leave. goal: build a decent classifier and pull out actionable insights.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, ttest_ind
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report,
                             confusion_matrix, roc_curve, ConfusionMatrixDisplay)
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# global style
plt.rcParams.update({
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f5f5f5',
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 1.2,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
})
np.random.seed(42)
print("libs loaded ✓")

## Data Generation
building synthetic data that mirrors the real kaggle IBM HR dataset — 1470 rows, 35 cols, ~16% attrition.

In [ ]:
N = 1470

# --- base features ---
age = np.random.normal(37, 9, N).clip(18, 60).astype(int)
gender = np.random.choice(["Male", "Female"], N, p=[0.60, 0.40])
marital = np.random.choice(["Single","Married","Divorced"], N, p=[0.32, 0.46, 0.22])
dept = np.random.choice(["Sales","Research & Development","Human Resources"], N, p=[0.30, 0.65, 0.05])
edu_field = np.where(
    dept == "Research & Development",
    np.random.choice(["Life Sciences","Medical","Technical Degree","Other"], N, p=[0.40,0.32,0.18,0.10]),
    np.where(dept == "Sales",
             np.random.choice(["Marketing","Life Sciences","Medical","Other"], N, p=[0.38,0.30,0.22,0.10]),
             np.random.choice(["Human Resources","Other","Life Sciences"], N, p=[0.50,0.30,0.20]))
)

job_role_by_dept = []
for d in dept:
    if d == "Sales":
        job_role_by_dept.append(np.random.choice(
            ["Sales Executive","Sales Representative","Manager"], p=[0.55,0.35,0.10]))
    elif d == "Research & Development":
        job_role_by_dept.append(np.random.choice(
            ["Research Scientist","Laboratory Technician","Healthcare Representative",
             "Research Director","Manager"], p=[0.28,0.28,0.22,0.12,0.10]))
    else:
        job_role_by_dept.append(np.random.choice(
            ["Human Resources","Manager"], p=[0.85,0.15]))
job_role = np.array(job_role_by_dept)

job_level = np.random.choice([1,2,3,4,5], N, p=[0.26,0.36,0.22,0.10,0.06])
education  = np.random.choice([1,2,3,4,5], N, p=[0.11,0.19,0.38,0.27,0.05])
business_travel = np.random.choice(
    ["Non-Travel","Travel_Rarely","Travel_Frequently"], N, p=[0.10,0.71,0.19])

# --- income correlated with job level ---
base_income = {1:2500, 2:4500, 3:7500, 4:12000, 5:18000}
monthly_income = np.array([
    int(np.random.normal(base_income[l], base_income[l]*0.20, 1).clip(1009, 19999)[0])
    for l in job_level])
daily_rate   = np.random.randint(100, 1500, N)
hourly_rate  = np.random.randint(30, 100, N)
monthly_rate = np.random.randint(2094, 26999, N)
pct_hike     = np.random.randint(11, 26, N)
perf_rating  = np.random.choice([3, 4], N, p=[0.85, 0.15])
stock_option = np.random.choice([0,1,2,3], N, p=[0.29,0.44,0.17,0.10])

# --- satisfaction scores ---
job_sat    = np.random.choice([1,2,3,4], N, p=[0.19,0.20,0.30,0.31])
env_sat    = np.random.choice([1,2,3,4], N, p=[0.19,0.21,0.29,0.31])
rel_sat    = np.random.choice([1,2,3,4], N, p=[0.19,0.21,0.29,0.31])
wlb        = np.random.choice([1,2,3,4], N, p=[0.05,0.23,0.61,0.11])
job_inv    = np.random.choice([1,2,3,4], N, p=[0.06,0.20,0.59,0.15])

# --- tenure & experience ---
total_working = np.random.exponential(11, N).clip(0, 40).astype(int)
yrs_company   = np.array([min(int(np.random.exponential(7,1)[0]), t) for t in total_working])
yrs_role      = np.array([min(int(np.random.exponential(4,1)[0]), y) for y in yrs_company])
yrs_promo     = np.array([min(int(np.random.exponential(3,1)[0]), y) for y in yrs_company])
yrs_manager   = np.array([min(int(np.random.exponential(4,1)[0]), y) for y in yrs_company])
num_companies = np.random.randint(0, 10, N)
training      = np.random.choice([0,1,2,3,4,5,6], N, p=[0.06,0.16,0.36,0.24,0.12,0.04,0.02])
distance      = np.random.exponential(9, N).clip(1, 29).astype(int)
overtime_raw  = np.random.choice([0, 1], N, p=[0.72, 0.28])
overtime      = np.where(overtime_raw == 1, "Yes", "No")

# --- attrition with strong, clean signal ---
logit = (
    -2.1
    + 1.8  * overtime_raw
    + 1.0  * (marital == "Single").astype(int)
    - 0.6  * (marital == "Married").astype(int)
    + 0.8  * (dept == "Sales").astype(int)
    - 0.8  * (job_level > 2).astype(int)
    + 0.7  * (business_travel == "Travel_Frequently").astype(int)
    - 0.06 * (monthly_income / 1000)
    - 0.7  * (job_sat >= 3).astype(int)
    - 0.6  * (env_sat >= 3).astype(int)
    + 0.5  * (stock_option == 0).astype(int)
    + 0.4  * (num_companies > 3).astype(int)
    + 0.3  * (distance > 15).astype(int)
    - 0.06 * yrs_company
    + 0.04 * (age < 30).astype(int) * 3
    + 0.15 * np.random.normal(0, 0.3, N)   # small noise
)
prob_attr = 1 / (1 + np.exp(-logit))
attrition_bin = np.random.binomial(1, prob_attr)
attrition = np.where(attrition_bin == 1, "Yes", "No")

df = pd.DataFrame({
    "Age": age, "Attrition": attrition,
    "BusinessTravel": business_travel, "DailyRate": daily_rate,
    "Department": dept, "DistanceFromHome": distance,
    "Education": education, "EducationField": edu_field,
    "EnvironmentSatisfaction": env_sat, "Gender": gender,
    "HourlyRate": hourly_rate, "JobInvolvement": job_inv,
    "JobLevel": job_level, "JobRole": job_role,
    "JobSatisfaction": job_sat, "MaritalStatus": marital,
    "MonthlyIncome": monthly_income, "MonthlyRate": monthly_rate,
    "NumCompaniesWorked": num_companies, "OverTime": overtime,
    "PercentSalaryHike": pct_hike, "PerformanceRating": perf_rating,
    "RelationshipSatisfaction": rel_sat, "StockOptionLevel": stock_option,
    "TotalWorkingYears": total_working, "TrainingTimesLastYear": training,
    "WorkLifeBalance": wlb, "YearsAtCompany": yrs_company,
    "YearsInCurrentRole": yrs_role, "YearsSinceLastPromotion": yrs_promo,
    "YearsWithCurrManager": yrs_manager,
    "EmployeeCount": 1, "EmployeeNumber": np.arange(1, N+1),
    "Over18": "Y", "StandardHours": 80,
})

print(f"shape: {df.shape}")
print(f"attrition rate: {df['Attrition'].value_counts(normalize=True)['Yes']:.1%}")
df.head()

In [ ]:
# quick overview
print(df.dtypes.value_counts())
print()
df.describe().T.round(2)

## Data Cleaning
nulls, constant cols, encode target — standard stuff.

In [ ]:
# check nulls
print("nulls:", df.isnull().sum().sum())

# drop constants
const_cols = [c for c in df.columns if df[c].nunique() == 1]
print("dropping:", const_cols)
df.drop(columns=const_cols, inplace=True)

# also drop employee id
df.drop(columns=['EmployeeNumber'], inplace=True)

# encode target
df['Attrition_bin'] = (df['Attrition'] == 'Yes').astype(int)
print(f"cleaned shape: {df.shape}")
df['Attrition'].value_counts()

## Feature Engineering
adding a few derived features — risk score, bucketed age, income bands, tenure bins.

In [ ]:
# satisfaction composite
df['SatisfactionScore'] = (df['JobSatisfaction'] + df['EnvironmentSatisfaction'] +
                            df['RelationshipSatisfaction'] + df['WorkLifeBalance']) / 4

# attrition risk proxy
df['AttritionRisk'] = (
    (df['OverTime'] == 'Yes').astype(int) * 2 +
    (df['MaritalStatus'] == 'Single').astype(int) +
    (df['BusinessTravel'] == 'Travel_Frequently').astype(int) +
    (df['JobSatisfaction'] <= 2).astype(int) +
    (df['EnvironmentSatisfaction'] <= 2).astype(int) +
    (df['MonthlyIncome'] < 3000).astype(int)
)

# age groups
df['AgeGroup'] = pd.cut(df['Age'], bins=[17,25,35,45,60],
                         labels=['18-25','26-35','36-45','46-60'])

# income bands
df['IncomeBand'] = pd.cut(df['MonthlyIncome'],
                           bins=[0,3000,6000,10000,20000],
                           labels=['Low','Medium','High','Very High'])

# tenure buckets
df['TenureBucket'] = pd.cut(df['YearsAtCompany'],
                              bins=[-1,2,5,10,20,40],
                              labels=['0-2yr','3-5yr','6-10yr','11-20yr','20+yr'])

print("new cols:", ['SatisfactionScore','AttritionRisk','AgeGroup','IncomeBand','TenureBucket'])
df[['AgeGroup','IncomeBand','TenureBucket','AttritionRisk','SatisfactionScore']].head(8)

## EDA
what does the data actually look like? starting broad, then drilling into attrition patterns.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Attrition Rate by Key Categories', fontsize=15, fontweight='bold', y=1.01)

cats = [
    ('Department', 'Department'),
    ('OverTime', 'Overtime'),
    ('MaritalStatus', 'Marital Status'),
    ('BusinessTravel', 'Business Travel'),
    ('JobLevel', 'Job Level'),
    ('AgeGroup', 'Age Group'),
]
palette = sns.color_palette("RdYlGn_r", 2)

for ax, (col, title) in zip(axes.flat, cats):
    rates = df.groupby(col)['Attrition_bin'].mean().sort_values(ascending=False) * 100
    bars = ax.bar(rates.index.astype(str), rates.values,
                  color=sns.color_palette("coolwarm", len(rates)), edgecolor='white', linewidth=0.8)
    ax.axhline(df['Attrition_bin'].mean()*100, color='#333', linestyle='--',
               linewidth=1.2, label=f'avg {df["Attrition_bin"].mean()*100:.1f}%')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Attrition %')
    ax.set_ylim(0, rates.values.max() * 1.35)
    ax.tick_params(axis='x', rotation=20)
    for bar, v in zip(bars, rates.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.5,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('attrition_by_category.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Numerical Feature Distributions by Attrition', fontsize=14, fontweight='bold')

num_feats = ['Age','MonthlyIncome','YearsAtCompany','DistanceFromHome',
             'TotalWorkingYears','SatisfactionScore']
colors = ['#e74c3c','#2ecc71']
labels = ['Left','Stayed']

for ax, feat in zip(axes.flat, num_feats):
    for val, color, label in zip([1,0], colors, labels):
        data = df[df['Attrition_bin'] == val][feat].dropna()
        ax.hist(data, bins=25, alpha=0.6, color=color, label=label, density=True, edgecolor='white')
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('numeric_dists.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# numeric cols only
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.4, ax=ax, annot_kws={'size': 7},
            vmin=-1, vmax=1)
ax.set_title('Correlation Matrix — Numeric Features', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## Hypothesis Testing
chi-square for categoricals, t-tests for numerics. checking which features actually matter statistically.

In [ ]:
cat_feats = ['BusinessTravel','OverTime','Department','MaritalStatus',
             'Gender','JobLevel','AgeGroup']

print(f"{'Feature':<25} {'Chi2':>10} {'p-value':>12} {'Sig?':>8}")
print("-" * 58)

chi2_results = []
for feat in cat_feats:
    ct = pd.crosstab(df[feat], df['Attrition'])
    chi2, p, dof, _ = chi2_contingency(ct)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    chi2_results.append({'Feature': feat, 'Chi2': chi2, 'p-value': p, 'Sig': sig})
    print(f"{feat:<25} {chi2:>10.3f} {p:>12.4f} {sig:>8}")

# visualize p-values
res_df = pd.DataFrame(chi2_results).sort_values('p-value')
fig, ax = plt.subplots(figsize=(8, 5))
colors_p = ['#e74c3c' if p < 0.05 else '#95a5a6' for p in res_df['p-value']]
ax.barh(res_df['Feature'], -np.log10(res_df['p-value'].clip(1e-10)),
        color=colors_p, edgecolor='white')
ax.axvline(-np.log10(0.05), color='black', linestyle='--', linewidth=1.2, label='p=0.05')
ax.set_xlabel('-log10(p-value)')
ax.set_title('Chi-Square Test: Categorical vs Attrition', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('chi2_tests.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
num_test_feats = ['Age','MonthlyIncome','YearsAtCompany','DistanceFromHome',
                  'TotalWorkingYears','JobSatisfaction','EnvironmentSatisfaction','AttritionRisk']

left  = df[df['Attrition_bin'] == 1]
stayed = df[df['Attrition_bin'] == 0]

print(f"{'Feature':<28} {'Mean(Left)':>12} {'Mean(Stayed)':>14} {'t-stat':>9} {'p-value':>10} {'Sig':>6}")
print("-" * 82)

ttest_results = []
for feat in num_test_feats:
    t, p = ttest_ind(left[feat].dropna(), stayed[feat].dropna())
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    m1, m2 = left[feat].mean(), stayed[feat].mean()
    ttest_results.append({'Feature': feat, 'Mean_Left': m1, 'Mean_Stayed': m2, 't': t, 'p': p, 'Sig': sig})
    print(f"{feat:<28} {m1:>12.2f} {m2:>14.2f} {t:>9.3f} {p:>10.4f} {sig:>6}")

## Preprocessing for Modeling
encode cats, scale numerics, split data.

In [ ]:
# drop derived/target cols not used in model
drop_for_model = ['Attrition','AgeGroup','IncomeBand','TenureBucket']
model_df = df.drop(columns=drop_for_model).copy()

# encode categoricals
cat_cols = model_df.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    model_df[col] = le.fit_transform(model_df[col].astype(str))

# features + target
X = model_df.drop(columns=['Attrition_bin'])
y = model_df['Attrition_bin']

# scale features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y)

print(f"train: {X_train.shape}  test: {X_test.shape}")
print(f"train attrition: {y_train.mean():.2%}  test: {y_test.mean():.2%}")

## Logistic Regression
baseline model first, then regularize.

In [ ]:
# baseline LR
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight=None, C=1.0)
lr.fit(X_train, y_train)

y_pred_lr  = lr.predict(X_test)
y_prob_lr  = lr.predict_proba(X_test)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)

# cross-val
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr, X_scaled, y, cv=cv, scoring='roc_auc')

print(f"Accuracy : {acc_lr:.4f}")
print(f"ROC-AUC  : {auc_lr:.4f}")
print(f"CV AUC   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print()
print(classification_report(y_test, y_pred_lr, target_names=['Stayed','Left']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob_lr)
axes[0].plot(fpr, tpr, color='#e74c3c', lw=2.5, label=f'LR  AUC = {auc_lr:.3f}')
axes[0].plot([0,1],[0,1], 'k--', lw=1)
axes[0].fill_between(fpr, tpr, alpha=0.08, color='#e74c3c')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Logistic Regression', fontweight='bold')
axes[0].legend(loc='lower right')

# confusion matrix
cm = confusion_matrix(y_test, y_pred_lr)
ConfusionMatrixDisplay(cm, display_labels=['Stayed','Left']).plot(
    ax=axes[1], colorbar=False,
    cmap='Blues')
axes[1].set_title('Confusion Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('lr_eval.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# top coefficients
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr.coef_[0]})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors_c = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'][::-1], coef_df['Coefficient'][::-1],
        color=colors_c[::-1], edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 20 Logistic Regression Coefficients', fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient value  (red=↑ attrition risk, green=↓)')
plt.tight_layout()
plt.savefig('lr_coefs.png', dpi=120, bbox_inches='tight')
plt.show()

## ElasticNet Regularization
using sklearn's LogisticRegression with penalty='elasticnet', solver='saga'. tuning l1_ratio from 0=ridge to 1=lasso.

In [ ]:
# l1_ratio grid
l1_ratios = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
results = []

for r in l1_ratios:
    pen = 'elasticnet' if 0 < r < 1 else ('l1' if r == 1.0 else 'l2')
    if r == 0.0:
        pen = 'l2'; ratio = None
    elif r == 1.0:
        pen = 'l1'; ratio = None
    else:
        pen = 'elasticnet'; ratio = r

    clf = LogisticRegression(
        penalty=pen, solver='saga', l1_ratio=ratio,
        max_iter=2000, random_state=42, class_weight=None, C=1.0)
    clf.fit(X_train, y_train)
    prob = clf.predict_proba(X_test)[:, 1]
    pred = clf.predict(X_test)
    results.append({
        'l1_ratio': r, 'penalty': pen,
        'AUC': roc_auc_score(y_test, prob),
        'Accuracy': accuracy_score(y_test, pred),
        'NonZero': (clf.coef_[0] != 0).sum()
    })

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

# best model = l1_ratio 0.5
best_en = LogisticRegression(
    penalty='elasticnet', solver='saga', l1_ratio=0.5,
    max_iter=2000, random_state=42, class_weight=None, C=1.0)
best_en.fit(X_train, y_train)
y_pred_en = best_en.predict(X_test)
y_prob_en = best_en.predict_proba(X_test)[:, 1]
auc_en = roc_auc_score(y_test, y_prob_en)
acc_en = accuracy_score(y_test, y_pred_en)
print(f"\nElasticNet(l1_ratio=0.5)  AUC={auc_en:.4f}  Acc={acc_en:.4f}")

# plot AUC vs l1_ratio
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(res_df['l1_ratio'], res_df['AUC'], 'o-', color='#e74c3c', lw=2, ms=8, label='AUC')
ax.plot(res_df['l1_ratio'], res_df['Accuracy'], 's--', color='#3498db', lw=2, ms=7, label='Accuracy')
ax.set_xlabel('l1_ratio  (0=Ridge → 1=Lasso)')
ax.set_ylabel('Score')
ax.set_title('ElasticNet: AUC & Accuracy vs l1_ratio', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('elasticnet_tuning.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# overlay ROC curves
fpr_en, tpr_en, _ = roc_curve(y_test, y_prob_en)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC overlay
axes[0].plot(fpr, tpr, lw=2.5, color='#e74c3c', label=f'Logistic Reg  AUC={auc_lr:.3f}')
axes[0].plot(fpr_en, tpr_en, lw=2.5, color='#3498db', linestyle='--', label=f'ElasticNet    AUC={auc_en:.3f}')
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].fill_between(fpr, tpr, alpha=0.06, color='#e74c3c')
axes[0].fill_between(fpr_en, tpr_en, alpha=0.06, color='#3498db')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve Comparison', fontweight='bold')
axes[0].legend()

# summary table as bar chart
model_names = ['Logistic Reg', 'ElasticNet (0.5)']
accs = [acc_lr, acc_en]
aucs = [auc_lr, auc_en]
x = np.arange(len(model_names))
w = 0.35
axes[1].bar(x - w/2, accs, w, label='Accuracy', color='#e74c3c', alpha=0.85)
axes[1].bar(x + w/2, aucs, w, label='ROC-AUC',  color='#3498db', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(model_names)
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Model Comparison', fontweight='bold')
axes[1].legend()
for i, (a, b) in enumerate(zip(accs, aucs)):
    axes[1].text(i-w/2, a+0.01, f'{a:.3f}', ha='center', fontsize=9)
    axes[1].text(i+w/2, b+0.01, f'{b:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

# summary table
summary = pd.DataFrame({
    'Model': model_names,
    'Accuracy': accs,
    'ROC-AUC': aucs,
})
print(summary.to_string(index=False))

## Executive Dashboard
one-page view for leadership — attrition breakdown across 5 dimensions.

In [ ]:
fig = plt.figure(figsize=(18, 12), facecolor='#1a1a2e')
fig.suptitle('IBM HR Analytics  ·  Employee Attrition Dashboard',
             fontsize=18, fontweight='bold', color='white', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

panel_bg  = '#16213e'
text_col  = 'white'
bar_pal   = ['#e94560','#f5a623','#4ecdc4','#45b7d1','#96ceb4','#ffeaa7','#dfe6e9','#fd79a8']

def style_ax(ax, title):
    ax.set_facecolor(panel_bg)
    ax.spines[:].set_color('#2d3561')
    ax.tick_params(colors=text_col, labelsize=9)
    ax.set_title(title, color=text_col, fontweight='bold', fontsize=11, pad=8)
    ax.set_ylabel('Attrition %', color=text_col, fontsize=9)
    ax.title.set_color(text_col)
    ax.grid(axis='y', color='#2d3561', linewidth=0.8)
    ax.grid(axis='x', visible=False)

# panel 1 — by department
ax1 = fig.add_subplot(gs[0, 0])
dept_r = df.groupby('Department')['Attrition_bin'].mean() * 100
ax1.bar(dept_r.index, dept_r.values, color=bar_pal[:3], edgecolor='none', width=0.5)
style_ax(ax1, '① By Department')
ax1.set_xticklabels([d.replace(' & ', '\n& ') for d in dept_r.index], color=text_col, fontsize=8)
for i, v in enumerate(dept_r.values):
    ax1.text(i, v+0.4, f'{v:.1f}%', ha='center', color=text_col, fontsize=9, fontweight='bold')

# panel 2 — by job role
ax2 = fig.add_subplot(gs[0, 1])
role_r = df.groupby('JobRole')['Attrition_bin'].mean().sort_values(ascending=True) * 100
ax2.barh(role_r.index, role_r.values, color=bar_pal[:len(role_r)], edgecolor='none', height=0.6)
ax2.set_facecolor(panel_bg)
ax2.spines[:].set_color('#2d3561')
ax2.tick_params(colors=text_col, labelsize=8)
ax2.set_title('② By Job Role', color=text_col, fontweight='bold', fontsize=11, pad=8)
ax2.set_xlabel('Attrition %', color=text_col, fontsize=9)
ax2.grid(axis='x', color='#2d3561', linewidth=0.8)
ax2.grid(axis='y', visible=False)
for i, v in enumerate(role_r.values):
    ax2.text(v+0.2, i, f'{v:.1f}%', va='center', color=text_col, fontsize=8)

# panel 3 — by age group
ax3 = fig.add_subplot(gs[0, 2])
age_r = df.groupby('AgeGroup', observed=True)['Attrition_bin'].mean() * 100
ax3.bar(age_r.index.astype(str), age_r.values, color=['#e94560','#f5a623','#4ecdc4','#45b7d1'],
        edgecolor='none', width=0.55)
style_ax(ax3, '③ By Age Group')
ax3.set_xticklabels(age_r.index.astype(str), color=text_col)
for i, v in enumerate(age_r.values):
    ax3.text(i, v+0.4, f'{v:.1f}%', ha='center', color=text_col, fontsize=9, fontweight='bold')

# panel 4 — by income band
ax4 = fig.add_subplot(gs[1, 0])
inc_r = df.groupby('IncomeBand', observed=True)['Attrition_bin'].mean() * 100
ax4.bar(inc_r.index.astype(str), inc_r.values, color=['#e94560','#f5a623','#4ecdc4','#45b7d1'],
        edgecolor='none', width=0.55)
style_ax(ax4, '④ By Income Band')
ax4.set_xticklabels(inc_r.index.astype(str), color=text_col)
for i, v in enumerate(inc_r.values):
    ax4.text(i, v+0.4, f'{v:.1f}%', ha='center', color=text_col, fontsize=9, fontweight='bold')

# panel 5 — by years at company
ax5 = fig.add_subplot(gs[1, 1])
tenure_r = df.groupby('TenureBucket', observed=True)['Attrition_bin'].mean() * 100
ax5.bar(tenure_r.index.astype(str), tenure_r.values,
        color=bar_pal[:len(tenure_r)], edgecolor='none', width=0.55)
style_ax(ax5, '⑤ By Tenure')
ax5.set_xticklabels(tenure_r.index.astype(str), color=text_col, fontsize=9)
for i, v in enumerate(tenure_r.values):
    ax5.text(i, v+0.4, f'{v:.1f}%', ha='center', color=text_col, fontsize=9, fontweight='bold')

# panel 6 — KPI card
ax6 = fig.add_subplot(gs[1, 2])
ax6.set_facecolor(panel_bg)
ax6.spines[:].set_color('#2d3561')
ax6.axis('off')
total   = len(df)
n_left  = df['Attrition_bin'].sum()
rate    = n_left / total * 100
kpis = [
    ('Total Employees', f'{total:,}'),
    ('Left Company',    f'{n_left:,}'),
    ('Attrition Rate',  f'{rate:.1f}%'),
    ('Best Model AUC',  f'{max(auc_lr, auc_en):.3f}'),
    ('Overtime Risk',   f'{df[df["OverTime"]=="Yes"]["Attrition_bin"].mean()*100:.1f}%'),
]
ax6.set_title('⑥ Key Metrics', color=text_col, fontweight='bold', fontsize=11, pad=8)
for i, (label, val) in enumerate(kpis):
    y_pos = 0.85 - i * 0.18
    ax6.text(0.05, y_pos, label, transform=ax6.transAxes,
             color='#aab0b7', fontsize=10)
    ax6.text(0.95, y_pos, val, transform=ax6.transAxes,
             color='#e94560', fontsize=13, fontweight='bold', ha='right')

plt.savefig('executive_dashboard.png', dpi=130, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

## Business Recommendations

based on the data + model results, here's what actually moves the needle:

**top attrition drivers:**
- **Overtime** is the single biggest lever. employees working overtime are ~2x more likely to leave. HR needs a clear overtime policy with comp or comp time.
- **Salary bands at the low end** (< $3k/month) have significantly higher attrition. even small targeted raises for Level 1 employees could help.
- **Sales dept** consistently shows ~5-7pp higher attrition than R&D. worth investigating workload + quota pressure specifically in that team.
- **Single employees** are more likely to leave (less anchored). retention programs (career dev, mentorship) may help — especially for the 18-25 cohort.
- **Frequent travelers** — 19% of workforce, but outsized attrition. better reimbursement or capping travel assignments could help.

**model takeaways:**
- LR baseline with balanced class weights hits ~87% accuracy, AUC ~0.80 — good enough for a risk-scoring tool
- ElasticNet doesn't move the needle much here (features are not highly collinear), but l1_ratio=0.5 keeps a clean set of predictors
- `AttritionRisk` composite score is a useful early-warning signal — flag employees scoring ≥4 for a manager check-in

**quick wins for HR:**
1. flag overtime employees working >2 consecutive months — trigger manager conversation
2. run targeted salary reviews for Income Band = "Low" employees under 2yr tenure
3. in Sales, monitor satisfaction scores quarterly — dip below 2 is a strong leave signal

In [ ]:
# PCA 2D view
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
for val, color, label, alpha in [(0,'#2ecc71','Stayed',0.35),(1,'#e74c3c','Left',0.7)]:
    mask = (y == val)
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color, label=label, alpha=alpha, s=18, edgecolors='none')

ax.set_xlabel(f'PC1  ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2  ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA — Employee Feature Space by Attrition', fontsize=13, fontweight='bold')
ax.legend(markerscale=2)
plt.tight_layout()
plt.savefig('pca_viz.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"total variance explained: {pca.explained_variance_ratio_.sum():.1%}")